# OASIS-1 174-Subject Multitask Evaluation

This notebook summarizes the larger public-data evaluation for the single-MRI multitask pipeline:

```text
Single T1 MRI
-> anatomical representation / segmentation
-> dementia classification
-> cognitive prediction
```

This run was originally requested as a 200-subject experiment. The available balanced OASIS-1 subset with usable MRI + MMSE + CDR labels contained **174 subjects**, so the final experiment uses 174 subjects.


## Experiment Setup

- Dataset: OASIS-1 public substitute for ADNI
- Subjects: 174 total
- Split: 121 train / 26 validation / 27 test
- MRI input: one downsampled T1 MRI per subject
- Segmentation target: FSL-FAST tissue labels
- Dementia label: `CDR 0` vs `CDR > 0`
- Cognitive target: MMSE
- Model: 3D multitask UNet-style model


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

PROJECT = Path.cwd()
if not (PROJECT / "outputs").exists():
    PROJECT = PROJECT.parent

RESULT_DIR = PROJECT / "outputs" / "oasis1_200_eval"
PREDICTIONS = RESULT_DIR / "oasis1_demo_predictions.csv"
METRICS = RESULT_DIR / "oasis1_demo_metrics.csv"
CONFUSION = RESULT_DIR / "confusion_matrix.png"
SLICES = RESULT_DIR / "mri_fast_model_slices.png"

metrics = pd.read_csv(METRICS)
predictions = pd.read_csv(PREDICTIONS)


## Metrics By Split

The key number for dementia classification is **balanced accuracy**, not raw accuracy. Balanced accuracy matters because it penalizes a model that predicts only one class.


In [ ]:
display(metrics)


## Main Test-Set Result

For the held-out test split:

| Task | Metric | Result | Interpretation |
| --- | ---: | ---: | --- |
| Anatomical segmentation | Dice | 0.884 | Strong tissue/anatomy learning |
| Dementia classification | Accuracy | 0.519 | Raw accuracy is misleading |
| Dementia classification | Balanced accuracy | 0.500 | Not successful yet; chance-level class separation |
| Dementia classification | AUC | 0.758 | Some ranking signal may exist |
| Cognitive prediction | MMSE MAE | 2.62 | Baseline-level prediction, not final |
| Cognitive prediction | MMSE RMSE | 3.37 | Moderate average error |


## Prediction Table

Each row contains the subject ID, split, true clinical label, predicted dementia probability, true MMSE, predicted MMSE, and segmentation Dice.


In [ ]:
display(predictions.head(20))
print(f"Rows: {len(predictions)}")
print(predictions.groupby(["split", "true_label"]).size())


## Confusion Matrix

The confusion matrix shows why dementia detection is **not successful yet**. At the default 0.5 threshold, the model predicted every subject as control/non-demented.

![Confusion matrix](../outputs/oasis1_200_eval/confusion_matrix.png)


In [ ]:
display(Image(filename=str(CONFUSION)))


## MRI / FAST / Model Segmentation Slices

These visual outputs show the anatomical part of the pipeline:

1. T1 MRI slice
2. FSL-FAST tissue target
3. Model-predicted segmentation

![MRI FAST model slices](../outputs/oasis1_200_eval/mri_fast_model_slices.png)


In [ ]:
display(Image(filename=str(SLICES)))


## Interpretation

The larger OASIS run proves that the technical pipeline works end-to-end:

```text
MRI loading
-> FAST segmentation targets
-> 3D multitask model
-> segmentation output
-> dementia probability output
-> MMSE output
-> evaluation tables and images
```

The strongest result is the anatomical segmentation output, with test Dice around **0.884**.

The dementia classifier is not reliable yet because probabilities stayed near **0.494**, just below the 0.5 threshold, so every subject was classified as control. This produces chance-level balanced accuracy.

The MMSE head predicts near the cohort mean, giving test MAE around **2.62**, but it is not yet a strong individualized cognitive model.


## Professor-Facing Summary

This notebook is a larger public-data prototype of the UCSF-style single-MRI multitask strategy. It uses OASIS-1 instead of ADNI and predicts three outputs from one T1 MRI: tissue segmentation, dementia label, and MMSE. The anatomy/segmentation part works well, while the clinical prediction heads need better data, preprocessing, MedicalNet/covariate training, GPU compute, and tuning.
